# SoRL → Distillation

**Idea**: Instead of warmup→SoRL (which didn't work), try the reverse: SoRL→Distillation.

**Phase 1**: Standard SoRL v3 training with `zipf+ortho`, `temperature=1.0`, `n=4` rollouts.  
This establishes diverse abstract vocabulary and good abstraction placements.

**Phase 2 v1 (naive)**: Same model, reconfigure:
- `temperature=0.0` (deterministic abstraction choice)
- `alpha_zipf=0.0, alpha_ortho=0.0` (remove diversity pressure)
- `num_rollouts=1` (no search, just use current best)

Assumption: abstraction choice is now stable from Phase 1, so we can distill — training only on traj+abs loss without search variance.

**Phase 2 v2 (hard)**: Copy model as `ref_model`, freeze it. Use `ref_model` to generate abstractions, train student model on those fixed abstractions.

**Validation**: 
- Does accuracy improve in Phase 2?
- Does vocab stay diverse or collapse after removing zipf/ortho?

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time, json, copy

if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.trainer_ablate import SoRLTrainerv3, SoRLConfig
from sorl.sorl_trainer import (
    infer_insert_mask, expand_prompt_len, insert_tokens_with_padding,
    sorl_search, corrupt_abstract_tokens,
)
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cpu


In [2]:
# Initialize model + tokenizer
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load dataset
train_ds, val_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256), \
                   get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Train: 7473 | Val: 1319


In [3]:
# ============================================================
# Helper: Vocab collapse measurement
# ============================================================
from torch.utils.data import DataLoader

pad_token_id = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())
abs_vocab_size = int(model.vocab_sizes[-1].item())


def measure_vocab_usage(mdl, dataset, K=4, n_batches=5):
    """Count abstract token usage across a few batches to detect collapse."""
    mdl.eval()
    dl = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
    abs_counts = torch.zeros(abs_vocab_size, device=device)
    with torch.no_grad():
        for i, batch in enumerate(dl):
            if i >= n_batches:
                break
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            pl = batch["prompt_len"].to(device)
            insert_mask = infer_insert_mask(ids, K, attn)
            exp_pl = expand_prompt_len(pl, insert_mask)
            exp_data, exp_mask = insert_tokens_with_padding(
                ids, attn, insert_mask, mdl.vocab_sizes[0], pad_token_id,
            )
            search_data, _ = mdl.recursion(
                exp_data, exp_mask, max_iterations=2,
                memory_span_abs=1792, memory_span_traj=1792,
                temperature=0.0, prompt_len=exp_pl,
            )
            abs_tokens = search_data[search_data >= base_vocab] - base_vocab
            abs_counts.scatter_add_(0, abs_tokens.long(), torch.ones_like(abs_tokens, dtype=torch.float))
    mdl.train()
    total = abs_counts.sum().item()
    if total == 0:
        return {"n_used": 0, "top1_pct": 100.0, "top3_pct": 100.0}
    sorted_counts = abs_counts.sort(descending=True).values
    return {
        "n_used": int((abs_counts > 0).sum().item()),
        "top1_pct": (sorted_counts[0] / total * 100).item(),
        "top3_pct": (sorted_counts[:3].sum() / total * 100).item(),
    }


def snapshot_metrics(mdl, train_ds, val_ds, label=""):
    """Take a full snapshot: vocab usage + accuracy."""
    vu = measure_vocab_usage(mdl, train_ds)
    print(f"  [{label}] Vocab: {vu['n_used']}/{abs_vocab_size} used | "
          f"top1={vu['top1_pct']:.1f}% top3={vu['top3_pct']:.1f}%")
    return vu


print("Helpers defined.")

Helpers defined.


## Phase 1: SoRL Training (zipf + ortho, temp=1.0, n=4)

Standard v3 training to establish diverse vocabulary and good abstractions.

In [ ]:
# ============================================================
# Phase 1 Config — SoRL v3 with zipf + ortho (exploration phase)
# ============================================================
phase1_config = SoRLConfig(
    # SoRL search
    K=4,
    num_rollouts=4,
    max_iterations=2,
    temperature=1.0,           # stochastic rollouts for exploration
    memory_span_abs=1792,
    memory_span_traj=1792,

    # Loss weights — v3 contrastive + zipf + ortho for diversity
    alpha_traj=1.0,
    alpha_contrastive=1.0,
    gamma_contrastive=0.5,
    alpha_abs=0.5,
    alpha_soft_zipf=1.0,       # diversity: zipf prior on abstract tokens
    alpha_ortho=1.0,           # diversity: orthogonality of abstract embeddings
    alpha_info_gain=0.0,

    # Corruption
    corrupt_method="shuffle",
    corrupt_ratio=0.3,

    # Optimizer
    lr=1e-5,
    emb_lr_mult=1.0,
    weight_decay=0.01,
    warmup_steps=50,
    cooldown_frac=0.4,
    max_grad_norm=1.0,

    # Training
    batch_size=2,
    gradient_accumulation_steps=4,
    num_epochs=3,
    log_every=10,
    eval_every=99999,
    save_every=99999,
    eval_samples=100,
    output_dir="./ckpt/distill_phase1",
)

print("Phase 1 config:")
print(f"  zipf={phase1_config.alpha_soft_zipf}, ortho={phase1_config.alpha_ortho}")
print(f"  temp={phase1_config.temperature}, n_rollouts={phase1_config.num_rollouts}")
print(f"  epochs={phase1_config.num_epochs}")

In [ ]:
# ============================================================
# Phase 1: Train SoRL v3 (zipf + ortho)
# ============================================================

# Accuracy evaluator (simple wrapper)
def compute_accuracy_fn(mdl, tok, dataset, dev, num_samples):
    return evaluate_accuracy(mdl, tok, dataset, dev, num_samples=num_samples)

trainer_p1 = SoRLTrainerv3(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    compute_accuracy=compute_accuracy_fn,
    config=phase1_config,
)

# Pre-training snapshot
print("=== Pre-training snapshot ===")
pre_vocab = snapshot_metrics(model, train_ds, val_ds, label="pre-train")

print("\n=== Phase 1: SoRL v3 (zipf+ortho, temp=1.0) ===")
t0 = time.time()
phase1_history = trainer_p1.train()
phase1_time = time.time() - t0
print(f"Phase 1 done in {phase1_time/60:.1f} min")

# Post Phase 1 snapshot
print("\n=== Post Phase 1 snapshot ===")
p1_vocab = snapshot_metrics(model, train_ds, val_ds, label="post-P1")

## Phase 2 v1: Naive Distillation

Reconfigure the **same** trainer: `temp=0`, no zipf/ortho, `n=1`.  
The model's abstraction choices are now deterministic — we just refine traj+abs loss.

In [ ]:
# ============================================================
# Phase 2 v1: Naive Distillation — reconfigure same trainer
# ============================================================
# Key changes: temp=0, no zipf/ortho, n=1 (no search)

phase2v1_config = SoRLConfig(
    # SoRL search — deterministic, single rollout
    K=4,
    num_rollouts=1,            # no search — just use current best
    max_iterations=2,
    temperature=0.0,           # deterministic abstraction choice
    memory_span_abs=1792,
    memory_span_traj=1792,

    # Loss weights — NO diversity pressure
    alpha_traj=1.0,
    alpha_contrastive=1.0,
    gamma_contrastive=0.5,
    alpha_abs=0.5,
    alpha_soft_zipf=0.0,       # OFF — no zipf
    alpha_ortho=0.0,           # OFF — no ortho
    alpha_info_gain=0.0,

    # Corruption (keep contrastive signal)
    corrupt_method="shuffle",
    corrupt_ratio=0.3,

    # Optimizer — lower LR for fine-tuning
    lr=5e-6,
    emb_lr_mult=1.0,
    weight_decay=0.01,
    warmup_steps=20,
    cooldown_frac=0.4,
    max_grad_norm=1.0,

    # Training
    batch_size=2,
    gradient_accumulation_steps=4,
    num_epochs=2,              # shorter — just refinement
    log_every=10,
    eval_every=99999,
    save_every=99999,
    eval_samples=100,
    output_dir="./ckpt/distill_phase2v1",
)

# Reconfigure the trainer in-place (model weights carry over from Phase 1)
trainer_p2v1 = SoRLTrainerv3(
    model=model,               # same model, Phase 1 weights
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    compute_accuracy=compute_accuracy_fn,
    config=phase2v1_config,
)

print("Phase 2 v1 config:")
print(f"  zipf={phase2v1_config.alpha_soft_zipf}, ortho={phase2v1_config.alpha_ortho}")
print(f"  temp={phase2v1_config.temperature}, n_rollouts={phase2v1_config.num_rollouts}")
print(f"  lr={phase2v1_config.lr}, epochs={phase2v1_config.num_epochs}")

print("\n=== Phase 2 v1: Distillation (temp=0, no zipf/ortho, n=1) ===")
t0 = time.time()
phase2v1_history = trainer_p2v1.train()
phase2v1_time = time.time() - t0
print(f"Phase 2 v1 done in {phase2v1_time/60:.1f} min")

# Post Phase 2 v1 snapshot
print("\n=== Post Phase 2 v1 snapshot ===")
p2v1_vocab = snapshot_metrics(model, train_ds, val_ds, label="post-P2v1")

## Phase 2 v2: Distillation with Frozen Reference Model

Copy the Phase 1 model as `ref_model` (frozen). Use it to generate abstractions deterministically,
then train the student model on those fixed abstractions. This decouples abstraction generation
from the model being trained.

In [ ]:
# ============================================================
# Phase 2 v2: Distillation with frozen ref_model
# ============================================================
# NOTE: This cell re-initializes from scratch to do a clean comparison.
#       If you've already run Phase 2 v1 above, re-run from Cell 2 first,
#       OR skip this cell and come back to it later.
#
# The idea:
#   ref_model (frozen, Phase 1 weights) generates abstract tokens
#   student model (trainable, also starts from Phase 1 weights) trains on them
#   ref_model uses temp=0, n=1 — deterministic abstractions
#   student only updates traj+abs loss, no search cost

import copy
from sorl.sorl_trainer import select_best_sequences


def distill_step_v2(student, ref_model, batch, cfg, base_vocab, pad_token_id, device):
    """
    One distillation step:
    1. ref_model generates abstract tokens (no grad, deterministic)
    2. student trains on those fixed abstractions
    """
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    prompt_len = batch["prompt_len"].to(device)

    # --- ref_model generates abstractions (frozen, deterministic) ---
    with torch.no_grad():
        best_data, _, _, exp_mask, exp_pl = sorl_search(
            ref_model, input_ids, attention_mask, prompt_len, pad_token_id,
            n=1, K=cfg["K"], max_iterations=cfg["max_iters"],
            memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"],
            temperature=0.0,  # deterministic
        )

    # --- Corrupt for contrastive signal ---
    total_vocab = int(ref_model.vocab_sizes.sum().item())
    corrupted_data = corrupt_abstract_tokens(
        best_data, base_vocab, total_vocab,
        method=cfg["corrupt_method"], corrupt_ratio=cfg["corrupt_ratio"],
    )

    # --- Student forward on ref_model's abstractions (WITH grad) ---
    outputs = student(
        input_ids=best_data, attention_mask=exp_mask,
        memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"],
    )
    logits = outputs.logits
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = best_data[..., 1:].contiguous()
    shift_attn = exp_mask[..., 1:].contiguous().clone()

    # Mask prompt
    seq_idx = torch.arange(shift_attn.size(1), device=device).unsqueeze(0)
    shift_attn[seq_idx < (exp_pl.unsqueeze(1) - 1)] = 0

    levels = (best_data >= base_vocab).long()[:, 1:]
    traj_mask = (levels == 0).float() * shift_attn.float()
    abs_mask = (1 - (levels == 0).float()) * shift_attn.float()
    loss_fct = nn.CrossEntropyLoss(reduction='none')

    # traj loss
    traj_logits = shift_logits.clone()
    traj_logits[..., base_vocab:] = -float("inf")
    safe_traj = shift_labels.clone()
    safe_traj[~traj_mask.bool()] = 0
    traj_per_tok = loss_fct(traj_logits.view(-1, traj_logits.size(-1)), safe_traj.view(-1))
    traj_loss = (traj_per_tok.view(best_data.shape[0], -1) * traj_mask).sum() / traj_mask.sum().clamp(min=1)

    # abs loss
    abs_logits = shift_logits.clone()
    abs_logits[..., :(base_vocab + 1)] = -float("inf")
    safe_abs = shift_labels.clone()
    safe_abs[~abs_mask.bool()] = base_vocab + 1
    abs_per_tok = loss_fct(abs_logits.view(-1, abs_logits.size(-1)), safe_abs.view(-1))
    abs_loss = (abs_per_tok.view(best_data.shape[0], -1) * abs_mask).sum() / abs_mask.sum().clamp(min=1)

    # Contrastive hinge (corrupted uses student forward, no grad on corruption)
    with torch.no_grad():
        c_out = student(
            input_ids=corrupted_data, attention_mask=exp_mask,
            memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"],
        )
        c_logits = c_out.logits[..., :-1, :].contiguous()
        c_traj_logits = c_logits.clone()
        c_traj_logits[..., base_vocab:] = -float("inf")
        c_safe = shift_labels.clone()
        c_safe[~traj_mask.bool()] = 0
        c_traj = loss_fct(c_traj_logits.view(-1, c_traj_logits.size(-1)), c_safe.view(-1))
        c_traj_loss = (c_traj.view(best_data.shape[0], -1) * traj_mask).sum() / traj_mask.sum().clamp(min=1)

    hinge_loss = (cfg["gamma"] + traj_loss - c_traj_loss).clamp(min=0)

    loss = cfg["alpha_traj"] * traj_loss + cfg["alpha_abs"] * abs_loss + cfg["alpha_contrastive"] * hinge_loss

    return loss, traj_loss.detach(), abs_loss.detach(), hinge_loss.detach()


print("distill_step_v2 defined.")

In [ ]:
# ============================================================
# Phase 2 v2: Training loop with frozen ref_model
# ============================================================
# To run this cleanly, you need Phase 1 weights in `model`.
# We deep-copy into ref_model (frozen) and train `model` (student).

ref_model = copy.deepcopy(model).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False
print(f"ref_model frozen: {sum(p.numel() for p in ref_model.parameters() if p.requires_grad)} trainable params")

v2_cfg = dict(
    K=4, max_iters=2,
    mem_span_abs=1792, mem_span_traj=1792,
    corrupt_method="shuffle", corrupt_ratio=0.3,
    alpha_traj=1.0, alpha_abs=0.5, alpha_contrastive=1.0, gamma=0.5,
    lr=5e-6, emb_lr_mult=1.0,
    weight_decay=0.01, max_grad_norm=1.0,
    warmup_steps=20,
    num_epochs=2, grad_accum=4, log_every=10,
    batch_size=2,
)

# Optimizer
emb_params = [p for n, p in model.named_parameters() if "embed_tokens" in n or "lm_head" in n]
other_params = [p for n, p in model.named_parameters() if "embed_tokens" not in n and "lm_head" not in n]
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": v2_cfg["lr"]},
    {"params": emb_params, "lr": v2_cfg["lr"] * v2_cfg["emb_lr_mult"]},
], weight_decay=v2_cfg["weight_decay"])

dl = DataLoader(train_ds, batch_size=v2_cfg["batch_size"], shuffle=True, collate_fn=collate_fn)
total_steps = len(dl) * v2_cfg["num_epochs"] // v2_cfg["grad_accum"]

v2_history = {"step": [], "loss": [], "traj": [], "abs": [], "hinge": []}
model.train()
global_step = 0
optimizer.zero_grad(set_to_none=True)

print(f"\n=== Phase 2 v2: Distillation with ref_model ===")
print(f"Total steps: {total_steps}")

t0 = time.time()
for epoch in range(v2_cfg["num_epochs"]):
    for batch_idx, batch in enumerate(dl):
        # LR warmup
        frac = min(global_step / max(v2_cfg["warmup_steps"], 1), 1.0)
        lr = v2_cfg["lr"] * frac
        optimizer.param_groups[0]["lr"] = lr
        optimizer.param_groups[1]["lr"] = lr * v2_cfg["emb_lr_mult"]

        loss, traj_l, abs_l, hinge_l = distill_step_v2(
            model, ref_model, batch, v2_cfg, base_vocab, pad_token_id, device,
        )

        (loss / v2_cfg["grad_accum"]).backward()

        if (batch_idx + 1) % v2_cfg["grad_accum"] == 0:
            if v2_cfg["max_grad_norm"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), v2_cfg["max_grad_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if (batch_idx + 1) % v2_cfg["log_every"] == 0:
            elapsed = time.time() - t0
            print(f"ep {epoch + (batch_idx+1)/len(dl):.2f} | step {global_step}/{total_steps} "
                  f"| loss={loss.item():.4f} traj={traj_l.item():.4f} "
                  f"abs={abs_l.item():.4f} hinge={hinge_l.item():.4f} | lr={lr:.2e}")
            v2_history["step"].append(global_step)
            v2_history["loss"].append(loss.item())
            v2_history["traj"].append(traj_l.item())
            v2_history["abs"].append(abs_l.item())
            v2_history["hinge"].append(hinge_l.item())

        del loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"=== Epoch {epoch+1} complete ===")

print(f"\nPhase 2 v2 done in {(time.time()-t0)/60:.1f} min")
p2v2_vocab = snapshot_metrics(model, train_ds, val_ds, label="post-P2v2")

## Evaluation & Comparison

In [ ]:
# ============================================================
# Accuracy Evaluation — NL vs K=4 at each phase
# ============================================================
# Run this after whichever Phase 2 variant you tested.
# For a full comparison, you'd checkpoint after Phase 1 and eval both v1 and v2.

model.eval()

print("Evaluating NL accuracy (no abstract tokens)...")
nl_result = evaluate_accuracy(model, tokenizer, val_ds, device, num_samples=100)
print(f"  NL accuracy: {nl_result['accuracy']*100:.1f}%")

print(f"Evaluating K=4 accuracy (with abstract tokens)...")
k4_result = evaluate_accuracy(model, tokenizer, val_ds, device, num_samples=100, eval_K=4)
print(f"  K=4 accuracy: {k4_result['accuracy']*100:.1f}%")

gap = (k4_result['accuracy'] - nl_result['accuracy']) * 100
print(f"\n  Gap (K=4 - NL): {gap:+.1f}pp")

model.train()

In [ ]:
# ============================================================
# Visualization: Phase 1 vs Phase 2 comparison
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- Phase 1 loss curves ---
ax = axes[0, 0]
if phase1_history.get("step"):
    ax.plot(phase1_history["step"], phase1_history["loss"], label="total", alpha=0.7)
    ax.plot(phase1_history["step"], phase1_history["traj_loss"], label="traj", alpha=0.7)
    ax.plot(phase1_history["step"], phase1_history["abs_loss"], label="abs", alpha=0.7)
ax.set_title("Phase 1: SoRL (zipf+ortho, temp=1.0)")
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# --- Phase 2 loss curves (pick whichever ran) ---
ax = axes[0, 1]
hist2 = v2_history if v2_history.get("step") else phase2v1_history if 'phase2v1_history' in dir() else {}
label2 = "Phase 2 v2 (ref_model)" if v2_history.get("step") else "Phase 2 v1 (naive)"
if hist2.get("step"):
    ax.plot(hist2["step"], hist2["loss"], label="total", alpha=0.7)
    ax.plot(hist2["step"], hist2["traj"], label="traj", alpha=0.7)
    ax.plot(hist2["step"], hist2["abs"], label="abs", alpha=0.7)
ax.set_title(f"{label2}: Distillation")
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# --- Vocab collapse comparison ---
ax = axes[1, 0]
labels = ["Pre-train", "Post P1"]
n_used = [pre_vocab["n_used"], p1_vocab["n_used"]]
top3   = [pre_vocab["top3_pct"], p1_vocab["top3_pct"]]

# Add whichever Phase 2 ran
if 'p2v1_vocab' in dir() and p2v1_vocab:
    labels.append("Post P2v1")
    n_used.append(p2v1_vocab["n_used"])
    top3.append(p2v1_vocab["top3_pct"])
if 'p2v2_vocab' in dir() and p2v2_vocab:
    labels.append("Post P2v2")
    n_used.append(p2v2_vocab["n_used"])
    top3.append(p2v2_vocab["top3_pct"])

x = np.arange(len(labels))
ax.bar(x - 0.2, n_used, 0.35, label="# tokens used", color="steelblue")
ax.bar(x + 0.2, top3, 0.35, label="top-3 %", color="coral")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.axhline(y=90, color='r', linestyle='--', alpha=0.3, label='collapse (90%)')
ax.axhline(y=abs_vocab_size, color='b', linestyle='--', alpha=0.3, label=f'max vocab ({abs_vocab_size})')
ax.set_title("Vocab Diversity Across Phases")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# --- Summary ---
ax = axes[1, 1]
ax.axis('off')
final_v = p2v2_vocab if 'p2v2_vocab' in dir() and p2v2_vocab else \
          p2v1_vocab if 'p2v1_vocab' in dir() and p2v1_vocab else p1_vocab
collapsed = final_v["top1_pct"] > 90
verdict = "COLLAPSED" if collapsed else "DIVERSE"
color = "red" if collapsed else "green"
summary = (
    f"SoRL -> Distillation Results\n"
    f"{'='*32}\n\n"
    f"Phase 1 vocab: {p1_vocab['n_used']}/{abs_vocab_size}\n"
    f"Phase 1 top3:  {p1_vocab['top3_pct']:.1f}%\n\n"
    f"Final vocab:   {final_v['n_used']}/{abs_vocab_size}\n"
    f"Final top1:    {final_v['top1_pct']:.1f}%\n"
    f"Final top3:    {final_v['top3_pct']:.1f}%\n\n"
    f"Verdict: {verdict}\n\n"
    f"NL acc:  {nl_result['accuracy']*100:.1f}%\n"
    f"K=4 acc: {k4_result['accuracy']*100:.1f}%\n"
    f"Gap:     {gap:+.1f}pp"
)
ax.text(0.05, 0.5, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor=color, alpha=0.15))

plt.tight_layout()
plt.savefig("./figure/sorl_distill.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to ./figure/sorl_distill.png")

# SoRL → Distillation

**Idea**: Instead of warmup→SoRL (which didn't work), try the reverse: SoRL→Distillation.

**Phase 1**: Standard SoRL v3 training with `zipf+ortho`, `temperature=1.0`, `n=4` rollouts.  
This establishes diverse abstract vocabulary and good abstraction placements.

**Phase 2 v1 (naive)**: Same model, reconfigure:
- `temperature=0.0` (deterministic abstraction choice)
- `alpha_zipf=0.0, alpha_ortho=0.0` (remove diversity pressure)
- `num_rollouts=1` (no search, just use current best)

Assumption: abstraction choice is now stable from Phase 1, so we can distill — training only on traj+abs loss without search variance.

**Phase 2 v2 (hard)**: Copy model as `ref_model`, freeze it. Use `ref_model` to generate abstractions, train student model on those fixed abstractions.

**Validation**: 
- Does accuracy improve in Phase 2?
- Does vocab stay diverse or collapse after removing zipf/ortho?